In [2]:

# ==========================================
# CELL 1: PROJECT SETUP
# ==========================================

import os
import sys
import torch
import numpy as np
import pandas as pd
import random
import time

from pathlib import Path

# Project directory
PROJECT_DIR = Path(r"E:\Cotton-Disease-Project")

# Set working directory
os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Create output directories
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
RESULT_DIR = PROJECT_DIR / "results"
GRAPH_DIR = RESULT_DIR / "graphs"

for folder in [CHECKPOINT_DIR, RESULT_DIR, GRAPH_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Import our custom model
from models.morph_cnn import MORPH_CNN

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PROJECT SETUP SUCCESSFUL")
print("Project:", PROJECT_DIR)
print("Model file:", (PROJECT_DIR / "models/morph_cnn.py").is_file())
print("Device:", device)
print("PyTorch:", torch.__version__)


PROJECT SETUP SUCCESSFUL
Project: E:\Cotton-Disease-Project
Model file: True
Device: cpu
PyTorch: 2.14.1+cpu


In [3]:

# ==========================================
# CELL 2: DATASET INITIALIZATION
# ==========================================

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

DATA_DIR = (
    PROJECT_DIR / "data" / "raw" /
    "Cotton_Original_Dataset"
)

BATCH_SIZE = 16
IMAGE_SIZE = 224

assert DATA_DIR.is_dir(), f"Dataset not found: {DATA_DIR}"

# Training transformations
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# Validation and testing transformations
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# Read original dataset
base_dataset = datasets.ImageFolder(str(DATA_DIR))

classes = base_dataset.classes
targets = np.array(base_dataset.targets)

# Stratified split: 70 / 15 / 15
indices = np.arange(len(targets))

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    stratify=targets,
    random_state=SEED
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=targets[temp_idx],
    random_state=SEED
)

# Separate transform pipelines
train_dataset = datasets.ImageFolder(
    str(DATA_DIR),
    transform=train_transform
)

eval_dataset = datasets.ImageFolder(
    str(DATA_DIR),
    transform=eval_transform
)

train_data = Subset(train_dataset, train_idx)
val_data = Subset(eval_dataset, val_idx)
test_data = Subset(eval_dataset, test_idx)

# Data loaders
train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

print("DATASET READY")
print("Classes:", classes)
print("Training:", len(train_data))
print("Validation:", len(val_data))
print("Testing:", len(test_data))
print("Train batches:", len(train_loader))
print("Device:", device)

images, labels = next(iter(train_loader))

print("Image batch:", images.shape)
print("Label batch:", labels.shape)


DATASET READY
Classes: ['Alternaria Leaf Spot', 'Bacterial Blight', 'Fusarium Wilt', 'Healthy Leaf', 'Verticillium Wilt']
Training: 961
Validation: 206
Testing: 206
Train batches: 61
Device: cpu
Image batch: torch.Size([16, 3, 224, 224])
Label batch: torch.Size([16])


In [4]:

# ==========================================
# CELL 3: MORPH-CNN TRAINING
# ==========================================

import torch.nn as nn
import torch.optim as optim

# Training configuration
EPOCHS = 30
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
PATIENCE = 7

# Initialize model
model = MORPH_CNN(num_classes=len(classes)).to(device)

# Calculate class weights using training samples only
train_labels = targets[train_idx]

class_counts = np.bincount(
    train_labels,
    minlength=len(classes)
)

class_weights = len(train_labels) / (
    len(classes) * class_counts
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

# History
history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "learning_rate": []
}

best_val_loss = float("inf")
best_val_acc = 0.0
early_stop_counter = 0

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_model.pth"

# ==========================================
# TRAINING FUNCTION
# ==========================================

def train_one_epoch():

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


# ==========================================
# VALIDATION FUNCTION
# ==========================================

def validate():

    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


# ==========================================
# MAIN TRAINING LOOP
# ==========================================

print("MORPH-CNN TRAINING STARTED")
print("=" * 75)

start_time = time.time()

for epoch in range(EPOCHS):

    epoch_start = time.time()

    train_loss, train_acc = train_one_epoch()

    val_loss, val_acc = validate()

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)
    history["learning_rate"].append(current_lr)

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc*100:.2f}% | "
        f"LR: {current_lr:.6f} | "
        f"Time: {time.time()-epoch_start:.1f}s"
    )

    # Save best validation checkpoint
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_val_acc = val_acc

        early_stop_counter = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "best_val_accuracy": best_val_acc,
                "classes": classes,
                "class_to_idx": base_dataset.class_to_idx,
                "seed": SEED
            },
            BEST_MODEL_PATH
        )

        print("  Best model checkpoint saved!")

    else:

        early_stop_counter += 1

    if early_stop_counter >= PATIENCE:

        print("Early stopping triggered!")
        break


# ==========================================
# SAVE TRAINING RESULTS
# ==========================================

history_df = pd.DataFrame(history)

history_path = RESULT_DIR / "training_history.csv"

history_df.to_csv(
    history_path,
    index=False
)

elapsed = time.time() - start_time

print("\n" + "=" * 75)
print("MORPH-CNN TRAINING COMPLETED")
print("Best Validation Loss:", best_val_loss)
print("Best Validation Accuracy:", best_val_acc * 100, "%")
print("Training Time:", round(elapsed / 60, 2), "minutes")
print("Best Model:", BEST_MODEL_PATH)
print("History:", history_path)


MORPH-CNN TRAINING STARTED
Epoch [01/30] | Train Loss: 1.0529 | Train Acc: 65.24% | Val Loss: 1.5534 | Val Acc: 63.59% | LR: 0.001000 | Time: 141.8s
  Best model checkpoint saved!
Epoch [02/30] | Train Loss: 0.8591 | Train Acc: 70.45% | Val Loss: 1.0069 | Val Acc: 66.50% | LR: 0.001000 | Time: 120.9s
  Best model checkpoint saved!
Epoch [03/30] | Train Loss: 0.7401 | Train Acc: 73.26% | Val Loss: 0.5747 | Val Acc: 81.55% | LR: 0.001000 | Time: 111.8s
  Best model checkpoint saved!
Epoch [04/30] | Train Loss: 0.7082 | Train Acc: 76.17% | Val Loss: 0.5988 | Val Acc: 80.58% | LR: 0.001000 | Time: 143.8s
Epoch [05/30] | Train Loss: 0.6243 | Train Acc: 78.67% | Val Loss: 0.9678 | Val Acc: 72.33% | LR: 0.001000 | Time: 107.4s
Epoch [06/30] | Train Loss: 0.6015 | Train Acc: 78.77% | Val Loss: 0.4074 | Val Acc: 85.44% | LR: 0.001000 | Time: 111.1s
  Best model checkpoint saved!
Epoch [07/30] | Train Loss: 0.5691 | Train Acc: 79.60% | Val Loss: 0.7366 | Val Acc: 75.73% | LR: 0.001000 | Time: 95